<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-03-prompting/lesson-3.1-system-prompts/notebooks/GCP_Capstone_3.1_System_Prompts.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.1 System Prompts & Generation Config
**Netsetos GenAI Engineering — GCP Capstone**

Persona design, ThinkingConfig, output caps, and A/B testing prompts.


## Setup


In [ ]:
!pip install -q google-genai==2.21.0 scipy
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS

from google import genai
from google.genai import types

client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')


## Cell 1: system_instruction — Your First Persona


In [ ]:
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='What is a Kubernetes pod?',
    config=types.GenerateContentConfig(
        system_instruction="""You are CloudArchitect, a senior GCP solutions architect
with 15 years of experience. Explain using everyday analogies.
Keep answers under 100 words. End with a practical tip.""",
        # temperature / top_p / top_k removed: gemini-3.6-flash ignores them (model page, GA 2026-07-21); Google recommends the default on every Gemini 3 model
        max_output_tokens=500,
        thinking_config=types.ThinkingConfig(thinking_level="LOW"),
    ),
)
print(response.text)
print(f'\nTokens: {response.usage_metadata.candidates_token_count}')


## Cell 2: Chat Session — Persona Persists


In [ ]:
chat = client.chats.create(
    model='gemini-3.6-flash',
    config=types.GenerateContentConfig(
        system_instruction='You are a pirate. Respond in pirate speak.',
        thinking_config=types.ThinkingConfig(thinking_level="LOW"),
    ),
)
r1 = chat.send_message('What is Python?')
print('Turn 1:', (r1.text or '')[:100])
r2 = chat.send_message('And Django?')
print('Turn 2:', r2.text[:100])


## Cell 3: Temperature Sweep


In [ ]:
# Myth-buster: the five outputs are effectively identical - 3.6 Flash ignores temperature
prompt = 'Suggest a name for an AI research assistant.'

outputs = []
for temp in [0.0, 0.3, 0.7, 1.0, 1.5]:
    r = client.models.generate_content(
        model='gemini-3.6-flash', contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temp, max_output_tokens=50,
            thinking_config=types.ThinkingConfig(thinking_level="LOW")),
    )
    outputs.append((r.text or '').strip()); print(f'  temp={temp:.1f} | {outputs[-1][:60]}')
print(f'  distinct outputs: {len(set(outputs))} of {len(outputs)}  (3.6 Flash ignores temperature - the demo passes at 1, fails visibly otherwise)')


## Cell 4: ThinkingConfig Comparison


In [ ]:
question = 'What is 17 * 23 + 456 - 89?'

for budget in ["LOW", "MEDIUM", "HIGH"]:   # Gemini 3.x thinks at a LEVEL, never off
    r = client.models.generate_content(
        model='gemini-3.6-flash', contents=question,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level=budget)),
    )
    think_tok = r.usage_metadata.thoughts_token_count or 0
    out_tok = r.usage_metadata.candidates_token_count or 0
    print(f'  budget={budget:<6} think={think_tok:<5} out={out_tok:<5} answer={(r.text or "").strip()[:40]}')


## Cell 5: 5-Section Persona — Research Analyst


In [ ]:
RESEARCH_ANALYST = """
<role>
You are ResearchBot, a senior AI/ML research analyst with 10 years experience.
</role>

<instructions>
1. Identify the core information need.
2. Provide evidence-based answers with citations.
3. Rate confidence: High / Medium / Low.
</instructions>

<constraints>
- Professional but accessible tone.
- Under 200 words unless asked for more.
- Say 'I don't have enough information' if uncertain.
</constraints>

<output_format>
## Summary\n[2-sentence overview]\n## Analysis\n[Details]\n## Sources\n[Citations]
</output_format>

<guardrails>
- Do NOT provide investment or legal advice.
- Flag conflicting information explicitly.
</guardrails>
"""

r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='What are the latest trends in RAG architectures?',
    config=types.GenerateContentConfig(
        system_instruction=RESEARCH_ANALYST,
        max_output_tokens=3000,
        thinking_config=types.ThinkingConfig(thinking_level="MEDIUM")),
)
print(r.text)


## Cell 6: India-Specific Personas


In [ ]:
BILINGUAL = """
<role>You are SahayakBot, a bilingual Hindi-English support agent.</role>
<instructions>
- Detect language from user message
- Hindi -> Devanagari with English tech terms
- Use formal 'aap' form. Currency in INR.
</instructions>
"""

for msg in ['What is UPI?', 'UPI kya hai?', 'mujhe payment issue hai']:
    r = client.models.generate_content(
        model='gemini-3.6-flash', contents=msg,
        config=types.GenerateContentConfig(
            system_instruction=BILINGUAL,
            max_output_tokens=300,
            thinking_config=types.ThinkingConfig(thinking_level="LOW")),
    )
    print(f'Q: {msg}\nA: {(r.text or '')[:100]}...\n')


## Cell 7: A/B Test Framework


In [ ]:
import time, statistics
from scipy import stats

def ab_test(client, variants, test_cases, runs=5):
    results = {}
    for name, sys_p in variants:
        results[name] = []
        for tc_in, kws in test_cases:
            for _ in range(runs):
                start = time.time()
                r = client.models.generate_content(
                    model='gemini-3.6-flash', contents=tc_in,
                    config=types.GenerateContentConfig(
                        system_instruction=sys_p,
                        max_output_tokens=500, seed=42,
                        thinking_config=types.ThinkingConfig(thinking_level="LOW")))
                lat = (time.time()-start)*1000
                txt = r.text or ''
                recall = sum(1 for k in kws if k.lower() in txt.lower())/len(kws)
                results[name].append({'recall':recall,'latency':lat})
    
    for name, data in results.items():
        recalls = [d['recall'] for d in data]
        print(f'{name}: recall={statistics.mean(recalls):.3f} +/- {statistics.stdev(recalls):.3f}')
    
    names = list(results.keys())
    if len(names) >= 2:
        a = [d['recall'] for d in results[names[0]]]
        b = [d['recall'] for d in results[names[1]]]
        if statistics.pvariance(a) == 0 and statistics.pvariance(b) == 0:
            print('p-value: n/a - identical recalls in both variants (seed=42 makes runs repeat); add test cases')
        else:
            _, p = stats.ttest_ind(a, b)
            print(f'p-value: {p:.4f} | Significant: {"YES" if p<0.05 else "NO"}')

variants = [
    ('concise', 'Be concise and factual.'),
    ('expert', 'You are a senior GCP architect. Give detailed answers.'),
]
tests = [
    ('What is Cloud Run?', ['serverless','container','scale']),
    ('What is a VPC?', ['network','virtual','private']),
]
ab_test(client, variants, tests)


## Cell 8: The prompt config module (the kit: services/rag-api/generator.py)


In [ ]:
# DocuMind prompt config - the lesson's module, the same 4 presets as the page; the lane's prompt lives in deploy/services/rag-api/generator.py.
# Knobs that matter on Gemini 3.x: persona, thinking budget, output cap - no sampling params.
RAG_CONFIG = types.GenerateContentConfig(
    # The SYSTEM prompt of the lane's own API (deploy/services/rag-api/generator.py), verbatim: rules 2, 3
    # and 5 are what the structured answer (3.2) and the live gate (4.8) depend on.
    system_instruction="""You are DocuMind, a retrieval-grounded assistant.
Rules:
1. Answer ONLY from the numbered context below. Never invent sources.
2. Cite using [N] where N is the chunk number. Multiple chunks: [1,2].
3. If the context does not contain the answer, set answerable=false and say so.
4. Keep answers under 300 words unless asked for more.
5. A quote is the clause that answers - at most twenty-five words, never a whole section.
""",
    max_output_tokens=2048,
    thinking_config=types.ThinkingConfig(thinking_level="LOW"),
)

ANALYSIS_CONFIG = types.GenerateContentConfig(
    system_instruction="""You are DocuMind Analyst, a research synthesis engine.
Compare, contrast, and synthesize information across multiple documents.
Use structured output with headers. Rate confidence: High/Medium/Low.""",
    max_output_tokens=4096,
    thinking_config=types.ThinkingConfig(thinking_level="MEDIUM"),
)

CREATIVE_CONFIG = types.GenerateContentConfig(
    system_instruction="""You are DocuMind Writer, a content creation assistant.
Generate engaging, original content. Use vivid language and varied structure.""",
    max_output_tokens=8192,
    thinking_config=types.ThinkingConfig(thinking_level="MEDIUM"),
)

CODE_CONFIG = types.GenerateContentConfig(
    system_instruction="""You are DocuMind Coder, a Python/GCP code assistant.
Write clean, documented, production-ready code. Include error handling.
Use type hints. Follow Google Python Style Guide.""",
    max_output_tokens=8192,
    thinking_config=types.ThinkingConfig(thinking_level="MEDIUM"),
)

CONFIGS = {
    'rag': RAG_CONFIG,
    'analysis': ANALYSIS_CONFIG,
    'creative': CREATIVE_CONFIG,
    'code': CODE_CONFIG,
}

def get_config(task_type: str) -> types.GenerateContentConfig:
    return CONFIGS.get(task_type, RAG_CONFIG)

# Test each
for task in CONFIGS:
    r = client.models.generate_content(
        model='gemini-3.6-flash', contents='Explain RAG in 2 sentences.', config=get_config(task))
    print(f'{task}: {(r.text or '')[:80]}...')


## ✅ Lesson 3.1 Complete!

- ✅ system_instruction inside GenerateContentConfig
- ✅ 5-section persona template (role/instructions/constraints/format/guardrails)
- ✅ Temperature sweep myth-buster: gemini-3.6-flash ignores temperature/top_p/top_k (verified 2026-09-03)
- ✅ ThinkingConfig budget control (0=lowest level, 4096 and 8192=larger budgets for reasoning — thinking is never fully off on 3.x)
- ✅ India-specific personas (regulatory, bilingual, cost optimizer)
- ✅ A/B testing framework with statistical significance
- ✅ The prompt config module with four presets (RAG / Analysis / Creative / Code) + get_config(); the lane's is deploy/services/rag-api/generator.py

**Next: Lesson 3.2 — Structured Output & JSON Mode**
